# Day 09. Exercise 01
# Gridsearch

## 0. Imports

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import joblib
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import ParameterGrid, cross_val_score
from tqdm.notebook import tqdm
import importlib
import tqdm.notebook as tqdm_notebook

## 1. Preprocessing

1. Read the file [`day-of-week-not-scaled.csv`](https://drive.google.com/file/d/1AlGvsJDSzPT_70caausx8bFuupIEZkfh/view?usp=sharing). It is similar to the one from the previous exercise, but this time we did not scale continuous features (we are not going to use logreg anymore). Don't forget to enrich the table with the 'dayofweek' column from the previous day's .csv-file.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [8]:
df = pd.read_csv("../data/day-of-week-not-scaled.csv")
old_df = pd.read_csv("../data/dayofweek.csv")
df["dayofweek"] = old_df["dayofweek"]
X = df.drop(columns=["dayofweek"])
y = df["dayofweek"]

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=21,
    stratify=y
)

## 2. SVM gridsearch

1. Using `GridSearchCV` try different parameters of kernel (`linear`, `rbf`, `sigmoid`), C (`0.01`, `0.1`, `1`, `1.5`, `5`, `10`), gamma (`scale`, `auto`), class_weight (`balanced`, `None`) use `random_state=21` and `probability=True` and get the best combination of them in terms of accuracy.
2. Create a dataframe from the results of the gridsearch and sort it ascendingly by the `rank_test_score`. Check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [10]:
param_grid = {
    "kernel": ["linear", "rbf", "sigmoid"],
    "C": [0.01, 0.1, 1, 1.5, 5, 10],
    "gamma": ["scale", "auto"],
    "class_weight": ["balanced", None],
}

model = SVC(
    random_state=21,
    probability=True
)

grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring="accuracy",
    cv=StratifiedKFold(n_splits=10),
    n_jobs=-1,
)

grid.fit(X_train, y_train)

/Users/a1111/Desktop/DSB11_ML_Advanced_ID_1577654-1/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/a1111/Desktop/DSB11_ML_Advanced_ID_1577654-1/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/a1111/Desktop/DSB11_ML_Advanced_ID_1577654-1/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/a1111/Desktop/DSB11_ML_Advanced_ID_1577

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",SVC(probabili...ndom_state=21)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'C': [0.01, 0.1, ...], 'class_weight': ['balanced', None], 'gamma': ['scale', 'auto'], 'kernel': ['linear', 'rbf', ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo...shuffle=False)
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support f

In [11]:
print(grid.best_params_)
print(grid.best_score_)

{'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf'}
0.8857490326147042


In [12]:
results = pd.DataFrame(grid.cv_results_)
results = results.sort_values("rank_test_score")
print(
    results[[
    "params",
    "mean_test_score",
    "std_test_score",
    "rank_test_score",
]].head(10)
)

                                               params  mean_test_score  \
70  {'C': 10, 'class_weight': None, 'gamma': 'auto...         0.885749   
64  {'C': 10, 'class_weight': 'balanced', 'gamma':...         0.873875   
58  {'C': 5, 'class_weight': None, 'gamma': 'auto'...         0.832349   
52  {'C': 5, 'class_weight': 'balanced', 'gamma': ...         0.820453   
66  {'C': 10, 'class_weight': None, 'gamma': 'scal...         0.728452   
69  {'C': 10, 'class_weight': None, 'gamma': 'auto...         0.728452   
60  {'C': 10, 'class_weight': 'balanced', 'gamma':...         0.720260   
63  {'C': 10, 'class_weight': 'balanced', 'gamma':...         0.720260   
48  {'C': 5, 'class_weight': 'balanced', 'gamma': ...         0.713577   
51  {'C': 5, 'class_weight': 'balanced', 'gamma': ...         0.713577   

    std_test_score  rank_test_score  
70        0.018843                1  
64        0.022551                2  
58        0.036143                3  
52        0.031530               

In [13]:
svm_grid = grid

Лучший результат получен при kernel="rbf", C=10, gamma="auto", class_weight=None: средняя accuracy на кросс-валидации — 0.88575, стандартное отклонение — 0.01884. С class_weight="balanced" при остальных тех же параметрах accuracy равна 0.87388 — разница составляет 1.19 процентного пункта. Лучший вариант с линейным ядром показал accuracy 0.72845, что ниже лучшего результата на 15.73 процентного пункта. В этом переборе более простая линейная модель заметно уступает RBF.


## 3. Decision tree

1. Using `GridSearchCV` try different parameters of `max_depth` (from `1` to `49`), `class_weight` (`balanced`, `None`) and `criterion` (`entropy` and `gini`) and get the best combination of them in terms of accuracy. Use `random_state=21`.
2. Create a dataframe from the results of the gridsearch and sort it ascendingly by the `rank_test_score`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [14]:
param_grid = {
    "max_depth": range(1, 50),
    "class_weight": ["balanced", None],
    "criterion": ["entropy", "gini"],
}

model = DecisionTreeClassifier(random_state=21)

tree_grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring="accuracy",
    cv=StratifiedKFold(n_splits=10)
)

tree_grid.fit(X_train, y_train)


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",DecisionTreeC...ndom_state=21)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'class_weight': ['balanced', None], 'criterion': ['entropy', 'gini'], 'max_depth': range(1, 50)}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo...shuffle=False)
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",Tr

In [15]:
print(tree_grid.best_params_)
print(tree_grid.best_score_)


{'class_weight': 'balanced', 'criterion': 'entropy', 'max_depth': 20}
0.8887064676616916


In [16]:
results = pd.DataFrame(tree_grid.cv_results_)
results = results.sort_values("rank_test_score")
print(
    results[[
    "params",
    "mean_test_score",
    "std_test_score",
    "rank_test_score",
]].head(10)
)


                                               params  mean_test_score  \
19  {'class_weight': 'balanced', 'criterion': 'ent...         0.888706   
40  {'class_weight': 'balanced', 'criterion': 'ent...         0.888701   
20  {'class_weight': 'balanced', 'criterion': 'ent...         0.888701   
37  {'class_weight': 'balanced', 'criterion': 'ent...         0.888701   
24  {'class_weight': 'balanced', 'criterion': 'ent...         0.888701   
25  {'class_weight': 'balanced', 'criterion': 'ent...         0.888701   
26  {'class_weight': 'balanced', 'criterion': 'ent...         0.888701   
35  {'class_weight': 'balanced', 'criterion': 'ent...         0.888701   
27  {'class_weight': 'balanced', 'criterion': 'ent...         0.888701   
29  {'class_weight': 'balanced', 'criterion': 'ent...         0.888701   

    std_test_score  rank_test_score  
19        0.015684                1  
40        0.016066                2  
20        0.015368                2  
37        0.016066               

Лучший результат получен при max_depth=20, class_weight='balanced', criterion="entropy": средняя accuracy на кросс-валидации — 0.88871, стандартное отклонение — 0.01568. Лучший вариант меньшей глубины (max_depth=19, class_weight='balanced', criterion="entropy") показал accuracy 0.88646 — разница составляет 0.22 процентного пункта. Более простое дерево даёт близкое качество.


## 4. Random forest

1. Using `GridSearchCV` try different parameters of `n_estimators` (`5`, `10`, `50`, `100`), `max_depth` (from `1` to `49`), `class_weight` (`balanced`, `None`) and `criterion` (`entropy` and `gini`) and get the best combination of them in terms of accuracy. Use random_state=21.
2. Create a dataframe from the results of the gridsearch and sort it ascendengly by the `rank_test_score`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [17]:
param_grid = {
    "n_estimators": [5, 10, 50, 100],
    "max_depth": range(1, 50),
    "class_weight": ["balanced", None],
    "criterion": ["entropy", "gini"],
}

model = RandomForestClassifier(random_state=21)

forest_grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring="accuracy",
    cv=StratifiedKFold(n_splits=10),
    n_jobs=-1,
)

forest_grid.fit(X_train, y_train)


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestC...ndom_state=21)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'class_weight': ['balanced', None], 'criterion': ['entropy', 'gini'], 'max_depth': range(1, 50), 'n_estimators': [5, 10, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo...shuffle=False)
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Sup

In [18]:
print(forest_grid.best_params_)
print(forest_grid.best_score_)


{'class_weight': None, 'criterion': 'entropy', 'max_depth': 32, 'n_estimators': 100}
0.919126589275843


In [19]:
results = pd.DataFrame(forest_grid.cv_results_)
results = results.sort_values("rank_test_score")
print(
    results[[
    "params",
    "mean_test_score",
    "std_test_score",
    "rank_test_score",
]].head(10).to_string(index=False)
)


                                                                              params  mean_test_score  std_test_score  rank_test_score
{'class_weight': None, 'criterion': 'entropy', 'max_depth': 42, 'n_estimators': 100}         0.919127        0.017439                1
{'class_weight': None, 'criterion': 'entropy', 'max_depth': 33, 'n_estimators': 100}         0.919127        0.017122                1
{'class_weight': None, 'criterion': 'entropy', 'max_depth': 34, 'n_estimators': 100}         0.919127        0.017439                1
{'class_weight': None, 'criterion': 'entropy', 'max_depth': 36, 'n_estimators': 100}         0.919127        0.017439                1
{'class_weight': None, 'criterion': 'entropy', 'max_depth': 37, 'n_estimators': 100}         0.919127        0.017439                1
{'class_weight': None, 'criterion': 'entropy', 'max_depth': 38, 'n_estimators': 100}         0.919127        0.017439                1
{'class_weight': None, 'criterion': 'entropy', 'max_dep

Лучший результат получен при n_estimators=100, max_depth=32, class_weight=None, criterion="entropy": средняя accuracy на кросс-валидации — 0.91913, стандартное отклонение — 0.01613. Лучший вариант с меньшим числом деревьев (n_estimators=50, max_depth=28, class_weight=None, criterion="gini") показал accuracy 0.91765 — разница составляет 0.15 процентного пункта. Модель с меньшим числом деревьев даёт близкое качество.


## 5. Progress bar

Gridsearch can be a quite long process and you may find yourself wondering when it will end.
1. Create a manual gridsearch for the same parameters values of random forest iterating through the list of the possible values and calculating `cross_val_score` for each combination. Try to increase `n_jobs`. The value `cv` for `cross_val_score` is 5.
2. Track the progress using the library `tqdm.notebook`.
3. Create a dataframe from the results of the gridsearch with the columns corresponding to the names of the parameters and `mean_accuracy` and `std_accuracy`.
4. Sort it descendingly by the `mean_accuracy`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [20]:
param_grid = {
    "n_estimators": [5, 10, 50, 100],
    "max_depth": range(1, 50),
    "class_weight": ["balanced", None],
    "criterion": ["entropy", "gini"],
}

In [21]:
importlib.reload(tqdm_notebook)


<module 'tqdm.notebook' from '/Users/a1111/Desktop/DSB11_ML_Advanced_ID_1577654-1/venv/lib/python3.14/site-packages/tqdm/notebook.py'>

In [22]:
manual_results = []

for params in tqdm(ParameterGrid(param_grid)):
    model = RandomForestClassifier(
        random_state=21,
        **params
    )

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        scoring="accuracy",
        cv=5,
        n_jobs=-1
    )

    manual_results.append({
        **params,
        "mean_accuracy": scores.mean(),
        "std_accuracy": scores.std()
    })


  0%|          | 0/784 [00:00<?, ?it/s]

In [23]:
manual_results = pd.DataFrame(manual_results)
manual_results = manual_results.sort_values("mean_accuracy", ascending=False)
print(manual_results.head(10))


    class_weight criterion  max_depth  n_estimators  mean_accuracy  \
698          NaN      gini         28            50       0.904290   
711          NaN      gini         31           100       0.904287   
783          NaN      gini         49           100       0.902806   
763          NaN      gini         44           100       0.902806   
751          NaN      gini         41           100       0.902806   
779          NaN      gini         48           100       0.902806   
739          NaN      gini         38           100       0.902806   
755          NaN      gini         42           100       0.902806   
735          NaN      gini         37           100       0.902806   
759          NaN      gini         43           100       0.902806   

     std_accuracy  
698      0.010961  
711      0.015204  
783      0.010460  
763      0.010460  
751      0.010460  
779      0.010460  
739      0.010460  
755      0.010460  
735      0.010460  
759      0.010460  


Лучший результат: 50 деревьев, max_depth=28, criterion="gini", class_weight=None — accuracy ≈ 0.90429. Вариант со 100 деревьями и max_depth=31 показывает практически такую же accuracy ≈ 0.904287. Поэтому можно выбрать более простую модель с 50 деревьями.

## 6. Predictions

1. Choose the best model and use it to make predictions for the test dataset.
2. Calculate the final accuracy.

In [24]:
best_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=32,
    criterion="entropy",
    class_weight=None,
    random_state=21
)

best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

print(f"Test accuracy: {accuracy_score(y_test, y_pred):.5f}")

Test accuracy: 0.93195
